In [1]:
import numpy as np
import sys
sys.path.append('..')
import os
import warnings
import pickle
warnings.filterwarnings("ignore")

from src.data_assemble.assemble_ml import *
from src.data_assemble.assemble_conv import *
from src.models.utils import *

In [2]:
path_to_files = ['../data/elev/*.tif', '../data/stash/WindProject/cmip_stash/*.nc']
filter_dict = {"years": ['2006'], "bands": ['max']}
rectangle_coords = {'lat_min': 43.5, 'lat_max': 50,'lon_min': 36.5, 'lon_max': 43}
target_res = {'lon_res': 0.25, 'lat_res': 0.25}

blocks = make_blocks(path_to_files, filter_dict, rectangle_coords, target_res, half_side_size = 3)

100%|██████████| 2/2 [00:00<00:00,  7.42it/s]


In [3]:
# feature_names = [['tasmax', 'tasmin', 'pr'], ['elevation']]
# path_to_tifs = ['../data/history/*.tif', '../data/elev/*.tif']
# loaded_arr = np.loadtxt("../wind_in_box.txt")  
# cmip = loaded_arr.reshape(3652, 15, 22)
# blocks = make_blocks(feature_names, path_to_tifs, cmip=cmip, half_side_size = 3, dset_num = 0)

Reading from .tifs
.tifs has been read


100%|██████████| 4/4 [00:00<00:00, 9208.13it/s]


In [4]:
start = '2006-01-01'
end = '2016-01-01'
df = pd.read_csv('../data_meteo_kk.csv')
station_list = pd.read_csv('../weatherstation_list.csv')

In [5]:
station_names = ['Анапа', 'Армавир', 'Краснодар, Круглик', 'Сочи', 'Туапсе', 'Приморско-Ахтарск', 'Красная Поляна']
stations_pixs = get_pixel_stations(path_to_tifs[0], feature_names[0], station_names, station_list)

In [6]:
target = get_y(df, start, end, speed_th=20)

In [7]:
X, y = assemble_numpy_ds(blocks, target, stations_pixs)

100%|██████████| 7/7 [00:00<00:00, 336.53it/s]


In [8]:
path_to_dump = os.path.join('..', 'data','nn_data')
trg_path = os.path.join(path_to_dump, 'target')
obj_path = os.path.join(path_to_dump, 'objects')
for k in X.keys():
    X_station = X[k]
    y_station = y[k]

    st_path = os.path.join(path_to_dump, k)
    if not os.path.isdir(st_path):
        os.makedirs(st_path)
    
    with open(os.path.join(st_path, 'objects.npy'),'wb') as f:
        # pickle.dump(X_station, f)
        np.save(f, X_station)
    with open(os.path.join(st_path, 'target.npy'),'wb') as f:
        # pickle.dump(y_station, f)
        np.save(f, y_station)
